#Algoritmo di Correzione per un Motore di Ricerca

# Caso d'Uso Aziendale: Searchify

## Introduzione all'Azienda
Searchify è una startup innovativa che offre un motore di ricerca personalizzato per aziende di medie dimensioni, consentendo di cercare informazioni specifiche nei loro database interni. Uno dei principali vantaggi di Searchify è la semplicità d'uso: gli utenti possono trovare rapidamente documenti, report e altre risorse aziendali digitando parole chiave. Tuttavia, l'azienda si trova ad affrontare un problema che mina l'esperienza dell'utente.

## Problema
Il problema principale di Searchify è che molti utenti commettono errori di digitazione durante l'inserimento delle parole chiave. Questi errori causano risultati di ricerca nulli o non pertinenti, portando a insoddisfazione tra gli utenti. Ad esempio, se un utente cerca "raporto vendite 2023" invece di "rapporto vendite 2023", il motore di ricerca non restituisce alcun risultato.

## Obiettivo del Progetto
L'obiettivo è sviluppare un algoritmo di correzione automatica per il motore di ricerca di Searchify. Questo algoritmo dovrà:

*   Rilevare automaticamente gli errori di digitazione o le parole non valide.
*   Suggerire la parola corretta più probabile.
*   Restituire risultati pertinenti basati sulla correzione suggerita.

L'implementazione di questa funzionalità migliorerà notevolmente l'esperienza utente, aumentando l'efficienza del motore di ricerca e la soddisfazione dei clienti.

## Benefici Attesi
*   **Miglioramento dell'accuratezza:** Riduzione degli errori di ricerca dovuti a digitazioni errate.
*   **Incremento della produttività:** Gli utenti troveranno le informazioni più velocemente.
*   **Aumento della fedeltà dei clienti:** Un'esperienza utente più fluida e soddisfacente porterà a una maggiore adozione del prodotto.

## Specifiche del Progetto
**Input:** Una stringa rappresentante una query di ricerca, digitata dall'utente.

**Output:**
*   Se la query contiene errori, il sistema suggerisce una correzione.
*   Se la query è corretta, restituisce la stessa query.

**Funzionalità Chiave:**
*   Confronto tra la parola inserita e un dizionario di parole corrette (da predefinire nel codice).
*   Implementazione di un algoritmo che calcoli la "distanza di edit" (es. distanza di Levenshtein) per trovare le parole più simili.
*   Gestione di casi d'uso realistici come errori di battitura comuni, lettere scambiate, omissioni o aggiunte.

## Consegna
Scrivi un programma Python che implementi l'algoritmo di correzione automatica per il motore di ricerca. Il tuo codice deve:

1.  Contenere una funzione principale chiamata `suggest_correction(query, dictionary)`.
    *   **Parametri:** `query` (stringa di input) e `dictionary` (lista di parole corrette).
    *   **Ritorno:** La parola corretta più probabile o la query originale se è già valida.
2.  Testare il funzionamento con almeno 10 casi d'uso (inclusi errori comuni e query corrette).
3.  Essere ben documentato, con commenti che spieghino le principali scelte implementative.
4.  Fornire un dizionario base contenente almeno 50 parole.

> **Nota:** Il progetto deve essere completato senza l'uso di librerie esterne per il calcolo della distanza di edit. L'obiettivo è implementare l'algoritmo interamente da zero.

------------------------

-------------------------------------------------------------------

# PARTE A — Versione minima (solo requisiti della consegna)


------------------------------------------------------------------------------------------------------

### A1 — Definizione del Dizionario Aziendale
Per iniziare si definisce il `ENTERPRISE_DICTIONARY`. Per soddisfare i requisiti del progetto, sono stati selezionati esattamente 50 termini unici relativi al contesto aziendale.

In [ ]:
# Dictionary (Requirement: exactly 50 unique single words)

ENTERPRISE_DICTIONARY = [
    "accompagnamento", "account", "analisi", "annuale", "aperti",
    "assistenza", "backup", "bilancio", "budget", "busta",
    "clienti", "agenti", "commerciali", "iso-9001", "contabilità",
    "contratti", "costi", "credenziali", "curriculum", "database",
    "dipendenti", "fatturato", "fatture", "ferie", "formazione",
    "fornitori", "gestione", "inventario", "magazzino", "manuale",
    "marketing", "mensile", "obiettivi", "ordini", "paga", "piano",
    "pagamenti", "performance", "machine-learning", "rapporto",
    "richiesta", "ripristino", "salario", "scorte", "server",
    "sicurezza", "sistema", "spedizioni", "ticket", "vendite"
]

In [ ]:
# TEST number of words in the list
print(f"Total unique single words: {len(set(ENTERPRISE_DICTIONARY))}")

Total unique single words: 50


### A2 — Acquisizione dell'Input
Questa funzione gestisce l'interazione con l'utente, permettendo l'inserimento della query di ricerca grezza che verrà poi elaborata dal sistema.

In [ ]:
# funtion that ask the user an input query
def get_user_query() -> str:
    """
    Prompts the user for the search query and returns it as a raw string
    """

    input_query = input("Searchify - Enter your search query: ")
    return input_query

### A3 — Pulizia e Tokenizzazione
Prima di analizzare gli errori, la query deve essere normalizzata. Si utilizza `unicodedata` per la coerenza dei caratteri accentati, convertendo tutto in minuscolo e applicando una regex per separare le parole mantenendo formati speciali come codici ISO o termini composti.

In [ ]:
# import standard libraries

import unicodedata
import string
import re

In [ ]:
# defining global variable

# percentage of error acceptable to consider the correction, tollerance or lenght, minimum lenght to discard a token
TOLLERANCE = 2
WORD_PERCENTAGE = 0.7
MINIMUM_LENGTH = min(len(word) for word in ENTERPRISE_DICTIONARY) - TOLLERANCE

# regex pattern to keep point and comma separated decimals and dash separated string with words and numbers
TOKEN_PATTERN = r"\d+(?:[.,]\d+)?|[A-Za-z0-9]+(?:-[A-Za-z0-9]+)*"

In [ ]:
# function to clean the raw query and create a list of the contained tokens

def clean_and_tokenize(input_query: str) -> list[str]:
    """
    Cleans and tokenizes the raw search query.

    1. Empty query check
    2. Unicode normalization (NFC) = unifies accented characters that might
       be encoded in two different ways (e.g., 'è' = 'e' + accent)
    3. Strip = Removal of leading/trailing whitespace
    4. Conversion to lowercase for case-insensitive comparison with the dictionary
    5. Token extraction via regex filter
    """

    # Empty Query: blocked before any other processing
    if not input_query or not input_query.strip():
        raise ValueError("The query cannot be empty.")

    # Unicode normalization + Trim leading/trailing whitespace + Lowercase conversion
    normalized = unicodedata.normalize("NFC", input_query).strip().lower()

    # regex filter
    tokens = re.findall(TOKEN_PATTERN, normalized)

    # if the query was only punctuation, it has now become [] so raise error
    if not tokens:
        raise ValueError(
            "The query contains only invalid characters or punctuation."
        )

    return tokens

In [ ]:
# TEST query cleaning

print(clean_and_tokenize("  Raporto   Vendite 2023?  "))    # -> "raporto vendite 2023"
print(clean_and_tokenize("RAPPORTO,vendite!!"))             # -> "rapporto vendite"

['raporto', 'vendite', '2023']
['rapporto', 'vendite']


### A4 — Classificazione dei Token
Non tutti i termini richiedono una correzione ortografica. Vengono identificati i numeri, le parole già presenti nel dizionario e i termini troppo brevi, isolando solo i candidati che necessitano del calcolo della distanza di edit.

In [ ]:
# function that classify cleaned tokens

def classify_tokens(tokens:list[str], dictionary:list[str]) -> list[dict]:
    """
    Analyzes and classifies tokens to determine which ones require correction.

    Returns a list of dictionaries containing the token and its classification:
    - 'correct': exists in dictionary
    - 'numeric': is a number or decimal
    - 'short': characters in word <= {MINIMUM_LENGHT}
    - 'to_verify': candidate for spell checking
    """
    analysis_results = []

    for token in tokens:
        # 1. Check if numeric
        if re.fullmatch(r'\d+(?:[.,]\d+)?', token):
            status = "numeric"
        # 2. Check if already correct (exact match in dictionary)
        elif token in dictionary:
            status = "correct"
        # 3. Check length
        elif len(token) <= MINIMUM_LENGTH:
            status = "short"
        # 4. Otherwise, mark for verification
        else:
            status = "to_verify"

        analysis_results.append({"token": token, "status": status})

    return analysis_results

In [ ]:
# TEST classification
sample_tokens = ['raporto', 'vendite', '2023', 'd']
results = classify_tokens(sample_tokens, ENTERPRISE_DICTIONARY)

print(results)
# Output example: [{'token': 'raporto', 'status': 'to_verify'}, ..
    # .. {'token': 'vendite', 'status': 'correct'}, {'token': '2023', 'status': 'numeric'}, {'token': 'd', 'status': 'short'}]

[{'token': 'raporto', 'status': 'to_verify'}, {'token': 'vendite', 'status': 'correct'}, {'token': '2023', 'status': 'numeric'}, {'token': 'd', 'status': 'short'}]


### A5 — Algoritmo di Levenshtein (Implementazione da Zero)
Sistema di calcolo della distanza di edit. Implementato tramite programmazione dinamica, conta il numero minimo di operazioni (inserimento, cancellazione, sostituzione) necessarie per trasformare le parole presenti nella query in quelle presenti nel dizionario.

In [ ]:

# Levenshtein Distance Calculation

def calculate_levenshtein_distance(s1: str, s2: str) -> int:
    """
    Calculates the standard Levenshtein distance between two strings.

    Args:
        s1: query string.
        s2: vocabulary string.
    Returns:
        The minimum number of edits (insertions, deletions, substitutions) to obtain s2 from s1.
    """
    len_s1, len_s2 = len(s1), len(s2)

    # Initialize matrix with zeros size = (len_s1 + 1)x(len_s2 + 1)
    matrix = [[0] * (len_s2 + 1) for row in range(len_s1 + 1)]

    # Populate first column and first row with increasing numbers 0 -> len +1
    for i in range(len_s1 + 1): matrix[i][0] = i
    for j in range(len_s2 + 1): matrix[0][j] = j

    # Calculate costs
    for i in range(1, len_s1 + 1):
        for j in range(1, len_s2 + 1):
            # Cost is 0 if characters match, 1 otherwise
            substitution_cost = 0 if s1[i - 1] == s2[j - 1] else 1

            #define the matrix of needed changes to obtain the same word with:
            matrix[i][j] = min(
                matrix[i - 1][j] + 1,        # Deletion -> if need move vertically
                matrix[i][j - 1] + 1,        # Insertion -> if need move horizontally
                matrix[i - 1][j - 1] + substitution_cost  # Substitution -> diagonals
            )

    #returns the last cell of the matrix
    return matrix[len_s1][len_s2]

### A6 — Ricerca dei Candidati e Gestione delle Omogeneità (Tie-break)
Il dizionario viene filtrato per lunghezza per ottimizzare le prestazioni. Se più parole hanno la stessa distanza minima, applichiamo una logica di tie-break basata sulla lunghezza del prefisso comune per suggerire la correzione più naturale.

In [ ]:
# find Levenshtein Distance candidates in the dictionary

def find_levenshtein_candidates(to_verify: str, dictionary: list[str]) -> list[dict]:
    """
    Filters the dictionary by length based on TOLLERANCE and calculates Levenshtein for candidates.

    Args:
        to_verify: word to check.
        dictionary: List of valid words.
    Returns:
        A sorted list of dictionaries with terms and their distances.
    """
    len_word = len(to_verify)
    results = []

    # Define length range words to check based on TOLLERANCE
    min_len = len_word - TOLLERANCE
    max_len = len_word + TOLLERANCE

    #check words in dictionary
    for dict_term in dictionary:
        # Length filter to avoid to check all the dictionary
        if min_len <= len(dict_term) <= max_len:
            # Calculate distance only for potential candidates
            distance = calculate_levenshtein_distance(to_verify, dict_term)
            results.append({"term": dict_term, "distance": distance})

    # Select the elements in dictionary[distance] and sort them by increasing value
    results.sort(key=lambda x: x["distance"])
    return results

In [ ]:
# TEST candidates
query = "3,14 12-AA ISO-9001 Machine-Learning 3,14 12-AA test!!!"
clean= clean_and_tokenize(query)
print(f"Query: \"{query}\"\n")
print(f"Clened query: {clean}\n")
for token in clean:
    print (f"Results for token: \"{token}\"")
    print(f"\t {find_levenshtein_candidates(token, ENTERPRISE_DICTIONARY)}")

Query: "3,14 12-AA ISO-9001 Machine-Learning 3,14 12-AA test!!!"

Clened query: ['3,14', '12', 'aa', 'iso-9001', 'machine-learning', '3,14', '12', 'aa', 'test']

Results for token: "3,14"
	 [{'term': 'paga', 'distance': 4}, {'term': 'busta', 'distance': 5}, {'term': 'costi', 'distance': 5}, {'term': 'ferie', 'distance': 5}, {'term': 'piano', 'distance': 5}, {'term': 'aperti', 'distance': 6}, {'term': 'backup', 'distance': 6}, {'term': 'budget', 'distance': 6}, {'term': 'agenti', 'distance': 6}, {'term': 'ordini', 'distance': 6}, {'term': 'scorte', 'distance': 6}, {'term': 'server', 'distance': 6}, {'term': 'ticket', 'distance': 6}]
Results for token: "12"
	 [{'term': 'paga', 'distance': 4}]
Results for token: "aa"
	 [{'term': 'paga', 'distance': 2}]
Results for token: "iso-9001"
	 [{'term': 'iso-9001', 'distance': 0}, {'term': 'ordini', 'distance': 7}, {'term': 'salario', 'distance': 7}, {'term': 'scorte', 'distance': 7}, {'term': 'server', 'distance': 7}, {'term': 'sistema', 'distance

In [ ]:
# funtion that skips numeric and short words, and suggest the best match implementing prefix check in ties results

def _get_common_prefix_length(str1: str, str2: str) -> int:
    """Calculates the length of the exact common prefix."""
    # Compare paired characters of the two strings until they differ
    for i, (c1, c2) in enumerate(zip(str1, str2)):
        if c1 != c2:
            return i
    return min(len(str1), len(str2))


def _resolve_tie(token: str, candidates: list[dict]) -> str:
    """Resolves ties by isolating minimum distances and sorting them by common prefix."""
    min_dist = candidates[0]["distance"]
    ties = [c for c in candidates if c["distance"] == min_dist]

    if len(ties) == 1:
        return ties[0]["term"]

    # Sort the ties based on the length of the common prefix (descending = best matches first)
    ties.sort(
        key=lambda x: _get_common_prefix_length(token, x["term"]), reverse=True
    )

    return ties[0]["term"]


def correction_with_numeric_handling(query: list[dict], dictionary: list[str]) -> str:
    """
    Takes the classified query tokens, maintains numbers,
    and suggests the most similar word correction.

    Correction Logic:
    - Distance must be <= {WORD_PERCENTAGE} of token length.
    - Minimum allowed distance threshold is {TOLLERANCE}.
    - In case of ties, the candidate with the longest common prefix is chosen.
    """
    corrected_query = []

    for item in query:
        token = item["token"]
        status = item["status"]

        if status in ("numeric", "correct"):
            corrected_query.append(token)
        elif status == "short":
            continue
        else:
            # Dynamic threshold: max(TOLLERANCE, WORD_PERCENTAGE of token length)
            dynamic_tolerance = max(TOLLERANCE, int(len(token) * WORD_PERCENTAGE))

            candidates = find_levenshtein_candidates(token, dictionary)

            if candidates and candidates[0]["distance"] <= dynamic_tolerance:
                #look for best condidate in ties
                best_term = _resolve_tie(token, candidates)
                corrected_query.append(best_term)
            else:
                corrected_query.append(token)

    return " ".join(corrected_query)

In [ ]:
# TEST
test_query = "raporot 2023 is-901"
clean = clean_and_tokenize(test_query)
classified_query = classify_tokens(clean, ENTERPRISE_DICTIONARY)

# Focusing on the ties
candidates = find_levenshtein_candidates(clean[0], ENTERPRISE_DICTIONARY)
min_dist = candidates[0]['distance']
ties = [c['term'] for c in candidates if c['distance'] == min_dist]

print(f"Ties for '{clean[0]}' (dist {min_dist}): {ties}")

result = correction_with_numeric_handling(classified_query, ENTERPRISE_DICTIONARY)

print(f"Originale: {test_query}")
print(f"Corretta:  {result}")
print()

# Detailed breakdown
for token in clean:
    print (f"Results for token: \"{token}\"")
    print(f"\t {find_levenshtein_candidates(token, ENTERPRISE_DICTIONARY)[:3]}")

Ties for 'raporot' (dist 3): ['rapporto']
Originale: raporot 2023 is-901
Corretta:  rapporto 2023 iso-9001

Results for token: "raporot"
	 [{'term': 'rapporto', 'distance': 3}, {'term': 'aperti', 'distance': 4}, {'term': 'account', 'distance': 5}]
Results for token: "2023"
	 [{'term': 'paga', 'distance': 4}, {'term': 'busta', 'distance': 5}, {'term': 'costi', 'distance': 5}]
Results for token: "is-901"
	 [{'term': 'iso-9001', 'distance': 2}, {'term': 'sistema', 'distance': 5}, {'term': 'aperti', 'distance': 6}]


### A7 — Funzione Orchestratrice `suggest_correction`

La funzione `suggest_correction` orchestra tutte le fasi del processo di correzione. Come richiesto dalla consegna del progetto, tale funzione integra le componenti precedentemente definite: pulizia e tokenizzazione dell'input, classificazione dei token, e applicazione della distanza di Levenshtein con una soglia di tolleranza dinamica. Quest'ultima viene utilizzata per determinare l'affidabilità di una correzione, mantenendo il termine originale se la somiglianza non è sufficiente.

In [ ]:
# core function of the project, collect the functionalities of the previus introduced functions

def suggest_correction(query: str, dictionary: list[str]) -> str:
    """
    function that orchestrates the correction logic.
    1. Cleans and tokenizes the input query.
    2. Classifies tokens (correct, numeric, short, to_verify).
    3. Applies Levenshtein distance with dynamic tolerance for correction.
    4. Rebuilds the final string.
    """
    try:
        # Cleaning
        tokens = clean_and_tokenize(query)

        # Classification
        classified_query = classify_tokens(tokens, dictionary)

        # Suggestion logic
        return correction_with_numeric_handling(classified_query, dictionary)

    except ValueError as e:
        # Returns the error message (e.g., empty query)
        return str(e)

### A8 — Suite di Test
Test su 10 casi d'uso rappresentativi, inclusi errori di battitura comuni, termini corretti, numeri e query vuote, per validare l'efficacia dell'algoritmo rispetto agli obiettivi di Searchify.

In [ ]:
def print_separator(length=80):
    print("-" * length)

def print_newline():
    print()

In [ ]:
# function dedicated to testing the 10 requeted cases

def tests():
    """
    Entry point for the Searchify correction algorithm.
    """
    print("=== Searchify ADVANCED Auto-Correction System (Part A) ===".center(80))

    test_inputs = [
        "rporot vendite 2023",              # words + numbers
        "busta pagaa4",                     # number typo
        "3,14",                             # Only number case
        "report mancine-learning",          # mix Italina English words
        "datase ISO-9001",                  # mix of numbers and letters
        "budgett stakeholder",   # Word not in the dictionary and not similar to any present ones
        "??",                               # Not accepted characters case
        "clienti aperti",                   # Correct case
        "ripristino servre",                # typo in letters
        "    "                              # Empty case
    ]

    print_separator()

    for raw_query in test_inputs:
        result = suggest_correction(raw_query, ENTERPRISE_DICTIONARY)
        print(f"Input:    '{raw_query}'")
        print(f"Output:   '{result}'")
        print_separator()

In [ ]:
tests()

           === Searchify ADVANCED Auto-Correction System (Part A) ===           
--------------------------------------------------------------------------------
Input:    'rporot vendite 2023'
Output:   'rporot vendite 2023'
--------------------------------------------------------------------------------
Input:    'busta pagaa4'
Output:   'busta paga'
--------------------------------------------------------------------------------
Input:    '3,14'
Output:   '3,14'
--------------------------------------------------------------------------------
Input:    'report mancine-learning'
Output:   'rapporto machine-learning'
--------------------------------------------------------------------------------
Input:    'datase ISO-9001'
Output:   'database iso-9001'
--------------------------------------------------------------------------------
Input:    'budgett stakeholder'
Output:   'budget stakeholder'
--------------------------------------------------------------------------------
Input:    '

-------------------------------------------------------

## PARTE B — Versione Potenziata (Searchify Advanced)

In questa sezione estendiamo l'algoritmo base per gestire scenari aziendali più complessi, superando i limiti della versione minima.



### B1 — Dizionario Strutturato e Multi-categoria
In questa versione avanzata, il dizionario non è più una semplice lista, ma una struttura a categorie (Dizionario di liste). Questo permette non solo di correggere la parola, ma di associare la query a un contesto aziendale specifico (es. Vendite, IT, HR), migliorando la pertinenza dei risultati.

In [ ]:
# Dictionary with composed words and categories

ENTERPRISE_DICTIONARY_IT = {
    "Vendite": [
        "rapporto", "previsioni commerciali", "performance", "obiettivi",
        "fatturato", "mensile", "annuale", "agente", "agenti", "trattativa",
        "vendita", "vendite", "commerciale", "commerciali", "marketing"
    ],
    "Amministrazione": [
        "fatture", "fattura", "fornitori", "clienti", "bilancio annuale", "pagamento",
        "analisi costi", "budget", "registro contabilità", "pagamenti"
    ],
    "Logistica": [
        "ordini aperti", "ordini chiusi", "inventario", "magazzino", "corriere",
        "bolle accompagnamento", "gestione scorte", "spedizioni", "iso-9001"
    ],
    "Risorse Umane (HR)": [
        "richiesta ferie", "busta paga", "contratto lavoro", "salario", "aumento",
        "curriculum candidato", "piano formazione", "valutazione dipendenti"
    ],
    "IT e Supporto": [
        "ticket assistenza", "credenziali database", "machine-learning", "account",
        "procedura sicurezza", "backup", "ripristino server", "software"
    ],
    "Ricerca Generale":[
        "informazioni generali", "performance", "risorse", "ricerca", "risultati"
    ]
}

### B2 — Analisi Dinamica del Dizionario (DictionaryAnalyzer)
Viene introdotta una classe dedicata per analizzare il dizionario. Rispetto alla Parte A, questa componente estrae automaticamente statistiche (parola più lunga/corta) e costruisce una mappa di termini 'flat' che include sia le singole parole che i termini composti, facilitando la ricerca.

In [ ]:
# objet to manage the operations done on the dictionary

class DictionaryAnalyzer:
    def __init__(self, dictionary: dict):
        self.dictionary = dictionary
        # Dynamically retrieve the variable name passed as an argument
        self.dictionary_name = self._get_variable_name(dictionary)
        self.words_per_category = {}
        self._analyze_and_build_dictionary()

    def _get_variable_name(self, target_obj) -> str:
        """
        Inspects global variables to find the original name
        of the dictionary object passed in input.
        """
        for name, value in globals().items():
            # Check if the memory reference matches exactly
            if value is target_obj:
                return name
        return "Unnamed Dictionary"

    def _analyze_and_build_dictionary(self):
        """
        Parses the dictionary, tokenizes compound phrases/words,
        extracts structured metrics, and builds a comprehensive flat list of all terms.
        """
        self.num_categories = len(self.dictionary)
        all_individual_words_for_stats = [] # Separate list for statistics
        all_unique_terms = set()            # Use a set to avoid duplicates
        term_to_category = {}

        # Changed regex: isolates alphanumeric blocks by cleanly separating with both space and hyphen
        # This allows for correct calculation of statistics on individual tokens (e.g., "iso", "9001")
        word_pattern = re.compile(r'[A-Za-zÀ-ù0-9]+')

        #create a lsit of unique terms made by single and space coposed words
        for category, words_list in self.dictionary.items():
            category_words_count = 0
            for item in words_list:
                item_lower = item.lower()
                # 1. Always add the original dictionary term  in the set() variable (e.g., "busta paga", "iso-9001", "machine-learning")
                all_unique_terms.add(item_lower)
                term_to_category.setdefault(item_lower, category)

                # Extraction of pure tokens for counting and statistics
                tokens = word_pattern.findall(item_lower)
                category_words_count += len(tokens)

                # 2. Sub-tokens always go into the searchable set, regardless of separator
                for token in tokens:
                    all_unique_terms.add(token)
                    term_to_category.setdefault(token, category)

                # 3. Statistics list (shortest/longest word) is stricter:
                #    - space-separated terms -> split, each token counts on its own
                #    - hyphenated or single terms -> kept as ONE block, sub-tokens excluded from stats
                if " " in item_lower:
                    all_individual_words_for_stats.extend(tokens)
                else:
                    all_individual_words_for_stats.append(item_lower)

            # Map the exact single-word count to the current category
            self.words_per_category[category] = category_words_count

        self.term_to_category = term_to_category                  # key = token, value = category

        self.total_words = len(all_individual_words_for_stats)    # number of individual words
        self.flat_dictionary = list(all_unique_terms)             # list of individual words and composed terms

        # Evaluate shortest and longest words from the fully tokenized list for stats
        if all_individual_words_for_stats:
            self.shortest_word = min(all_individual_words_for_stats, key=len)
            self.longest_word = max(all_individual_words_for_stats, key=len)
        else:
            self.shortest_word = None
            self.longest_word = None


    # funtions to obtain info about the dictionary

    def get_num_categories(self) -> int:
        return self.num_categories

    def get_num_words_per_category(self) -> dict:
        return self.words_per_category

    def get_total_words(self) -> int:
        return self.total_words

    def get_shortest_word(self) -> str:
        return self.shortest_word

    def get_longest_word(self) -> str:
        return self.longest_word

    def get_shortest_word_length(self) -> int:
        return len(self.shortest_word) if self.shortest_word else 0

    def get_longest_word_length(self) -> int:
        return len(self.longest_word) if self.longest_word else 0

    def get_flat_dictionary(self) -> list[str]:
            # Return a copy to prevent external modifications to the internal list
            return list(self.flat_dictionary)

    def get_topic(self, corrected_query: str) -> str:
      """
      Identifies the business category of a (corrected) query by checking
      whether any of its tokens map to a known dictionary term/category.
      Falls back to 'Ricerca Generale' if no token matches.
      """
      tokens = corrected_query.lower().split()
      for token in tokens:
          if token in self.term_to_category:
              return self.term_to_category[token]
      return "Ricerca Generale"


In [ ]:
# Initialize the analyzer object
analyzer = DictionaryAnalyzer(ENTERPRISE_DICTIONARY_IT)

In [ ]:
# TEST Analyzer

print(f"--- Dictionary {analyzer.dictionary_name} Analysis ---".center(80))
print_newline()
print(f"Number of categories: {analyzer.get_num_categories()}")
print_newline()
print("Words per category:")
for category, count in analyzer.get_num_words_per_category().items():
    print(f"  - {category}: {count} words")
print_newline()
print(f"Shortest word: '{analyzer.get_shortest_word()}' (length: {analyzer.get_shortest_word_length()})")
print(f"Longest word: '{analyzer.get_longest_word()}' (length: {analyzer.get_longest_word_length()})")
print_newline()
print(f"\nTotal words: {analyzer.get_total_words()}")


              --- Dictionary ENTERPRISE_DICTIONARY_IT Analysis ---              

Number of categories: 6

Words per category:
  - Vendite: 16 words
  - Amministrazione: 13 words
  - Logistica: 14 words
  - Risorse Umane (HR): 14 words
  - IT e Supporto: 13 words
  - Ricerca Generale: 6 words

Shortest word: 'paga' (length: 4)
Longest word: 'machine-learning' (length: 16)


Total words: 74


In [ ]:
# Advanced configuration variables

TOLLERANCE = 2
WORD_PERCENTAGE = 0.5
MINIMUM_LENGTH = len(analyzer.get_shortest_word()) - TOLLERANCE
COMPOUND_MERGE_THRESHOLD = 0.3  # More conservative threshold for compound word merging

print(f"Word percentage: {WORD_PERCENTAGE}")
print(f"Minimum word length: {MINIMUM_LENGTH}")
print(f"Tolerance: {TOLLERANCE}")
print(f"Compound Merge Threshold: {COMPOUND_MERGE_THRESHOLD}")

Word percentage: 0.5
Minimum word length: 2
Tolerance: 2
Compound Merge Threshold: 0.3


### B3 — Algoritmo di Damerau-Levenshtein
Rispetto a Levenshtein (Parte A), Damerau-Levenshtein riconosce lo **scambio di due lettere adiacenti** (trasposizione) come un singolo errore di battitura.

In [ ]:
# optimised version of the levenshtein funtion in which trasposition of subsequent letters counts as single error

def calculate_damerau_levenshtein(s1: str, s2: str) -> int:
    """
    Calculates the standard Damerau-Levenshtein distance between two strings.

    Args:
        s1: query string.
        s2: vocabulary string.
    Returns:
        The minimum number of edits (insertions, deletions, substitutions, tasposition) to obtain s2 from s1.
    """
    len_s1, len_s2 = len(s1), len(s2)

    # Initialize matrix with zeros size = (len_s1 + 1)x(len_s2 + 1)
    matrix = [[0] * (len_s2 + 1) for row in range(len_s1 + 1)]

    # Populate first column and first row with increasing numbers 0 -> len +1
    for i in range(len_s1 + 1): matrix[i][0] = i
    for j in range(len_s2 + 1): matrix[0][j] = j

    # Calculate costs
    for i in range(1, len_s1 + 1):
        for j in range(1, len_s2 + 1):
            # Cost is 0 if characters match, 1 otherwise
            substitution_cost = 0 if s1[i - 1] == s2[j - 1] else 1

            #define the matrix of needed changes to obtain the same word with:
            matrix[i][j] = min(
                matrix[i - 1][j] + 1,        # Deletion -> if need move vertically
                matrix[i][j - 1] + 1,        # Insertion -> if need move horizontally
                matrix[i - 1][j - 1] + substitution_cost  # Substitution -> diagonals
            )

# --------- Till here same as calculate_levenshtein_distance() -----------

            # Transposition check -> if letters are swapped, cost increases by 1 from the state before transposition
            if i > 1 and j > 1 and s1[i-1] == s2[j-2] and s1[i-2] == s2[j-1]:
                matrix[i][j] = min(matrix[i][j], matrix[i-2][j-2] + 1)

    return matrix[len_s1][len_s2]

In [ ]:
levenshtein = calculate_levenshtein_distance("evndite","vendite")
damerau_levenshtein = calculate_damerau_levenshtein("evndite","vendite")
print(f"Levenshtein distance = {levenshtein}")
print(f"Damerau Levenshtein distance = {damerau_levenshtein}")

Levenshtein distance = 2
Damerau Levenshtein distance = 1


### B4 — Gestione dei Termini Composti (Rolling Window)
La Parte A analizza i token singolarmente. La Parte B introduce il `rolling_window_merge`, che osserva coppie di parole adiacenti. Se 'bus' e 'pga' appaiono vicine, il sistema le unisce e le confronta con i termini composti del dizionario, gestendo anche errori in entrambe le parti (es. 'bus pga' -> 'busta paga').

In [ ]:
# funtions that checks if one of the two string passed are matched with a word in the whole dictionary (composed words and not)

def _try_exact_compound_merge(t1: str, t2: str, flat_dictionary: list[str]) -> tuple[str | None, int]:
    """
    Attempts to merge t1 and t2 if t1 is a known word and the combined form
    is an exact compound term in the dictionary.

    Returns:
        (merged_term, 2) if an exact compound match is found,
        (t1, 1) if t1 is a known word but no exact compound exists with t2
                (protects t1 from being altered by fuzzy merge attempts),
        (None, 0) if t1 is not a known single word at all.
    """
    if t1 in flat_dictionary:
        combined_str = t1 + t2
        for word in flat_dictionary:
            # Check if 'word' is a compound and if 'combined_str' matches its flattened form
            if " " in word and combined_str == word.replace(" ", ""):
                return word, 2  # Exact compound match found
        # t1 is a known word, but no exact compound with t2, so protect t1 from further merge attempts
        return t1, 1
    return None, 0  # t1 is not a known single word, so proceed to fuzzy merge


# funtions that checks if the two string passed are matched with a composed word in the dictionary

def _try_fuzzy_compound_merge(t1: str, t2: str, flat_dictionary: list) -> tuple[str | None, int]:
    """
    Attempts to merge t1 and t2 using fuzzy matching (Damerau-Levenshtein)
    against compound terms in the dictionary. This also handles transposed pairs
    (e.g. user typed the two words in the wrong order).

    Returns:
        (merged_term, 2) if a fuzzy compound match is found, else (None, 0).
    """
    for word in flat_dictionary:
        if " " in word:  # Only consider compound words from the dictionary (e.g., "piano formazione")
            parts = word.lower().split()
            if len(parts) == 2:
                p1, p2 = parts[0], parts[1]
                # Calculate separate tolerances for the two tokens
                thresh1 = max(TOLLERANCE, int(len(t1) * COMPOUND_MERGE_THRESHOLD))
                thresh2 = max(TOLLERANCE, int(len(t2) * COMPOUND_MERGE_THRESHOLD))

                # Check for direct order (t1 ~ p1, t2 ~ p2)
                if (calculate_damerau_levenshtein(t1, p1) <= thresh1 and
                    calculate_damerau_levenshtein(t2, p2) <= thresh2):
                    return word, 2  # Fuzzy compound match found (direct order)

                # Check for transposed order (t1 ~ p2, t2 ~ p1)
                if (calculate_damerau_levenshtein(t1, p2) <= thresh1 and
                    calculate_damerau_levenshtein(t2, p1) <= thresh2):
                    return word, 2  # Fuzzy compound match found (transposed order)
    return None, 0


In [ ]:
# function that checks two subsequent tokens in orther to match composed words

def rolling_window_merge(tokens: list[str], flat_dictionary: list[str]) -> list[str]:
    """
    Merges adjacent tokens only if their combination represents a real improvement
    compared to treating the tokens individually, preventing a correct word (e.g., 'informazioni')
    from being absorbed into an irrelevant compound term.
    """
    i = 0
    merged_tokens = []

    while i < len(tokens):
        if i + 1 < len(tokens):
            t1 = tokens[i]
            t2 = tokens[i + 1]

            # First, try to find an exact compound merge or protect 't1' if it's a known single word
            merged_term, consumed_count = _try_exact_compound_merge(t1, t2, flat_dictionary)

            # If no exact merge or protection occurred, try a fuzzy compound merge
            if consumed_count == 0:  # This means t1 was not found as a standalone or exact compound part
                merged_term, consumed_count = _try_fuzzy_compound_merge(t1, t2, flat_dictionary)

            if consumed_count > 0:  # If any merge (exact or fuzzy) or protection occurred (consumed 1 or 2 tokens)
                merged_tokens.append(merged_term)
                i += consumed_count
            else:  # No merge or protection found for the pair, process t1 alone
                merged_tokens.append(t1)
                i += 1
        else:  # Last token, no pair to merge with, append as is
            merged_tokens.append(tokens[i])
            i += 1

    return merged_tokens


In [ ]:
# Test Rolling Window Merge Functions
# This test verifies the logic for merging adjacent tokens into compound dictionary terms.

test_sequences = [
    (["bus", "pga"], "Fuzzy merge (both parts have typos)"),
    (["busta", "paga"], "Exact merge (both parts correct)"),
    (["informazioni", "generali"], "Protection (t1 is correct, prevents wrong merging)"),
    (["piano", "formazion"], "Fuzzy merge (typo in second part)"),
    (["paga", "busta"], "Transposed merge (tokens in reversed order)")
]

# Retrieve the flat dictionary from the analyzer
flat_dict = analyzer.get_flat_dictionary()

print(f"--- Rolling Window Logic Test ---")
print(f"{'INPUT TOKENS':<35} | {'MERGED RESULT':<25} | {'SCENARIO'}")
print_separator()
for tokens_in, scenario in test_sequences:
    # Execute the rolling window merge logic
    result = rolling_window_merge(tokens_in, flat_dict)
    print(f"{str(tokens_in):<35} | {str(result):<25} | {scenario}")

--- Rolling Window Logic Test ---
INPUT TOKENS                        | MERGED RESULT             | SCENARIO
--------------------------------------------------------------------------------
['bus', 'pga']                      | ['busta paga']            | Fuzzy merge (both parts have typos)
['busta', 'paga']                   | ['busta paga']            | Exact merge (both parts correct)
['informazioni', 'generali']        | ['informazioni generali'] | Protection (t1 is correct, prevents wrong merging)
['piano', 'formazion']              | ['piano', 'formazion']    | Fuzzy merge (typo in second part)
['paga', 'busta']                   | ['paga', 'busta']         | Transposed merge (tokens in reversed order)


### B5 — Tie-break Avanzato e Scoring Strategico

Nella Parte A, in caso di pareggio (stessa distanza di edit), l'algoritmo si basa solo sulla lunghezza del prefisso. Nella Parte B, è stato implementato un sistema di scoring più robusto:

1.  **Contatore dei caratteri in comune (`Counter`):** Utilizzando la classe `Counter` della libreria `collections` si analizza la frequenza di ogni singola lettera. Permettendo di capire quale candidato, a parità di errori, ha la "composizione materica" (set di lettere) più simile alla query originale.
2.  **Risoluzione delle omonimie:** La funzione `_determine_best_correction` agisce come un arbitro. Se più parole nel dizionario hanno la stessa distanza di Damerau-Levenshtein, viene premiata quella con il maggior numero di caratteri condivisi, riducendo così i falsi positivi.
3.  **Sistema di scoring:** La funzione `score_candidates` unifica il processo di scansione del dizionario, applicando filtri di lunghezza per l'efficienza e ordinando i risultati in modo che il miglior match sia sempre il primo della lista.

In [ ]:
# function to get list of best option for token substitution, compare the ties and choose
# the one with more similar chars to the token in input

from collections import Counter


def _get_shared_characters_count(str1: str, str2: str) -> int:
    """
    Computes the number of characters shared between two strings,
    taking into account the frequency of each letter (used as a
    tie-breaker between equally-distant candidates).
    """
    return sum((Counter(str1) & Counter(str2)).values())


def _determine_best_correction(token: str, candidates: list[dict], tolerance: int) -> str:
    """
    Determines the best correction for a token based on scored candidates,
    a dynamic tolerance, and tie-breaking by shared character count.
    """
    if not candidates:
        return token

    min_dist_for_token = candidates[0]["distance"]

    if min_dist_for_token > tolerance:
        return token

    ties = [c for c in candidates if c["distance"] == min_dist_for_token]

    if len(ties) > 1:
        # Apply shared characters count for tie-breaking
        ties.sort(key=lambda x: _get_shared_characters_count(token, x["term"]), reverse=True)

    return ties[0]["term"]

In [ ]:
# function to obtain an ordered dictionary based on best matches

def score_candidates(token: str, flat_dictionary: list[str], tolerance: int, restrict_to: list[str] | None = None) -> list[dict]:
    """
    Shared scoring helper: computes the Damerau-Levenshtein distance between
    `token` and every term in the dictionary (or in `restrict_to`, if given),
    and returns the candidates sorted by ascending distance.

    This consolidates the scan-and-score logic previously duplicated between
    `decompose_and_match` and the correction pipeline.

    Args:
        token: the query token to match.
        flat_dictionary: the flat list of dictionary terms.
        tolerance: the maximum allowed distance for a candidate to be considered.
        restrict_to: optional subset of terms to score against, instead of the
            full dictionary (used by decompose_and_match to pre-filter candidates
            that share alphabetic/numeric parts with the token).

    Returns:
        A list of {"term": ..., "distance": ...} dicts, sorted by distance.
    """
    if restrict_to is not None:
        pool = restrict_to
    else:
        len_token = len(token)
        min_len = len_token - tolerance
        max_len = len_token + tolerance
        # Length filter to avoid scanning the whole dictionary
        pool = [word for word in flat_dictionary if min_len <= len(word) <= max_len]

    candidates = [
        {"term": word, "distance": calculate_damerau_levenshtein(token, word)}
        for word in pool
    ]
    candidates.sort(key=lambda c: c["distance"])
    return candidates


In [ ]:
# tests for tie-breaking, character counting and scoring

# 1. Test _get_shared_characters_count
token_test = "pga"
cand1 = "paga"
cand2 = "pia"

count1 = _get_shared_characters_count(token_test, cand1)
count2 = _get_shared_characters_count(token_test, cand2)

# 2. Test _determine_best_correction (Tie-break scenario)
candidates = [
    {"term": cand1, "distance": calculate_damerau_levenshtein(token_test,cand1)},
    {"term": cand2, "distance": calculate_damerau_levenshtein(token_test,cand2)}
]

best = _determine_best_correction(token_test, candidates, tolerance=2)

# 3. Test score_candidates

flat_dict = analyzer.get_flat_dictionary()

scored_results = score_candidates(token_test, flat_dict, TOLLERANCE)

print(f"Damerau Levenshtein distance between '{token_test}' and '{candidates[0]['term']}' = {candidates[0]['distance']}")
print(f"Damerau Levenshtein distance between '{token_test}' and '{candidates[1]['term']}' = {candidates[1]['distance']}")
print_separator()
print(f"Shared chars between '{token_test}' and '{cand1}'= {count1}")
print(f"Shared chars between '{token_test}' and '{cand2}'= {count2}")
print_separator()
print(f"Best correction for '{token_test}' between '{candidates[0]['term']}' and '{candidates[1]['term']}'= {best}")
print_separator()

print(f"Candidates scoring for '{token_test}:")
print(f"\n{'TERM':<20} | {'DISTANCE':<10}")
print("-" * 35)

for res in scored_results:
    print(f"{res['term']:<20} | {res['distance']:<10}")


Damerau Levenshtein distance between 'pga' and 'paga' = 1
Damerau Levenshtein distance between 'pga' and 'pia' = 1
--------------------------------------------------------------------------------
Shared chars between 'pga' and 'paga'= 3
Shared chars between 'pga' and 'pia'= 2
--------------------------------------------------------------------------------
Best correction for 'pga' between 'paga' and 'pia'= paga
--------------------------------------------------------------------------------
Candidates scoring for 'pga:

TERM                 | DISTANCE  
-----------------------------------
paga                 | 1         
iso                  | 3         
piano                | 3         
9001                 | 4         
busta                | 4         
bolle                | 5         
costi                | 5         
ferie                | 5         


### B6 — Decomposizione Alfanumerica (Mixed Tokens)
Per codici come 'ISO-9001', gli utenti spesso dimenticano trattini o invertono i numeri. La funzione `decompose_and_match` separa la parte alfabetica da quella numerica, usando i numeri come 'ancora' affidabile per trovare il codice corretto nel dizionario, una funzione assente nella Parte A.

In [ ]:
# function to split token in its numeric and alphabetic parts, search for those in the dictionary and matches with candidates

def decompose_and_match(token: str, flat_dictionary: list[str], tolerance: int) -> str:
    """
    Handles mixed alphanumeric tokens (e.g., 'iso9001') by using the numeric part as an anchor.

    1. Splits the token into alphabetic and numeric components.
    2. Filters the dictionary for candidates containing that specific numeric sequence.
    3. Calculates Damerau-Levenshtein distance between the alphabetic parts of the
       token and the candidates to find the best match within the given tolerance.
    """
    alpha_part = "".join(re.findall(r'[a-zA-Z]+', token))
    num_part = "".join(re.findall(r'\d+', token))

    # If it's purely numeric or purely alpha, this specific logic might not be best,
    # but we still try to find codes containing the numeric sequence.
    if not num_part:
        return token

    # 2. Find candidates that contain the numeric part (common in technical codes)
    # or where the token is a significant substring.
    candidates_pool = [
        word for word in flat_dictionary
        if num_part in word.replace('-', '')
    ]

    if not candidates_pool:
        return token

    # 3. Score candidates using Damerau-Levenshtein distance
    scored = []
    for word in candidates_pool:
        word_alpha = "".join(re.findall(r'[a-zA-Z]+', word))
        # Compare only alpha-to-alpha: the numeric part is already anchored above
        dist = calculate_damerau_levenshtein(alpha_part, word_alpha) if alpha_part else 0
        scored.append({"term": word, "distance": dist})

    scored.sort(key=lambda c: c["distance"])
    if scored[0]["distance"] <= tolerance:
        return scored[0]["term"]

    return token

In [ ]:
# Test Decompose and Match Function
# This function isolates numeric anchors to fix technical codes or mixed tokens.

test_codes = [
    ("iso9001", "Missing dash in technical code"),
    ("19001", "Just numbers, no complete match"),
    ("iso9002", "No dash, no correct code"),
    ("9001iso", "Reversed parts in technical code"),
    ("busta2023", "Alpha-numeric mix (no dash in dict)"),
    ("isooo-9001", "Typo in the alpha part of a code")
]

# Retrieve the flat dictionary
flat_dict = analyzer.get_flat_dictionary()

print(f"--- Decompose and Match Logic Test ---")
print(f"{'INPUT TOKEN':<20} | {'MATCHED RESULT':<20} | {'SCENARIO'}")
print_separator()

for token, scenario in test_codes:
    # Using the global TOLLERANCE for consistency
    result = decompose_and_match(token, flat_dict, TOLLERANCE)
    print(f"{token:<20} | {result:<20} | {scenario}")

--- Decompose and Match Logic Test ---
INPUT TOKEN          | MATCHED RESULT       | SCENARIO
--------------------------------------------------------------------------------
iso9001              | iso-9001             | Missing dash in technical code
19001                | 19001                | Just numbers, no complete match
iso9002              | iso9002              | No dash, no correct code
9001iso              | iso-9001             | Reversed parts in technical code
busta2023            | busta2023            | Alpha-numeric mix (no dash in dict)
isooo-9001           | iso-9001             | Typo in the alpha part of a code


### B7 — Pipeline di Correzione Orchestrata
La nuova `suggest_correction_advanced` integra tutti i moduli precedenti. A differenza della Parte A, restituisce un oggetto strutturato che include sia la query corretta che il **Topic** (categoria aziendale), fornendo un output pronto per essere usato da un sistema di filtraggio dati.

In [ ]:
# advanced suggest_correctio() function

def suggest_correction_advanced(raw_query: str, analyzer) -> dict:
    """
    Enhanced correction pipeline:
    1. Cleaning and tokenization
    2. Rolling window merge for compound words
    3. Token classification (numeric / correct / short / to_verify)
    4. Correction, driven by classification:
       - numeric or correct -> token kept as is
       - short -> token dropped
       - to_verify -> decompose-and-match for mixed alphanumeric tokens,
         then Damerau-Levenshtein correction with dynamic tolerance
         (distance <= max(TOLLERANCE, WORD_PERCENTAGE * token length))
         and tie-breaking by shared character count; falls back to the
         original token if no candidate is close enough
    5. Topic association
    """
    try:
        flat_dictionary = analyzer.get_flat_dictionary()
        tokens = clean_and_tokenize(raw_query)

        # Process compound words
        tokens = rolling_window_merge(tokens, flat_dictionary)

        # Classify every token before deciding how to handle it
        classified_tokens = classify_tokens(tokens, flat_dictionary)


        final_query = []
        for item in classified_tokens:
            t = item["token"]
            status = item["status"]

            # Dynamic tolerance: scales with token length, with a hard floor
            dynamic_tolerance = max(TOLLERANCE, int(len(t) * WORD_PERCENTAGE))

            if status == "correct":
                final_query.append(t)
                continue

            if status == "numeric":
                decomposed_match = decompose_and_match(t, flat_dictionary, dynamic_tolerance)
                final_query.append(decomposed_match)
                continue

            if status == "short":
                # Drop tokens too short to be reliably corrected
                continue

            # status == "to_verify"

            # Attempt decompose and match for mixed tokens (e.g. "iso9001")
            decomposed_match = decompose_and_match(t, flat_dictionary, dynamic_tolerance)
            if decomposed_match != t:
                final_query.append(decomposed_match)
                continue

            candidates = score_candidates(t, flat_dictionary, dynamic_tolerance)
            best_term = _determine_best_correction(t, candidates, dynamic_tolerance)
            final_query.append(best_term)

        result_str = " ".join(final_query)
        return {
            "corrected": result_str,
            "topic": analyzer.get_topic(result_str)
        }
    except Exception as e:
        return {"error": str(e)}

In [ ]:
# test for the added features

def advanced_tests():
    print("=== SEARCHIFY ADVANCED TEST SUITE ===".center(80))
    cases = [
    "bus pga",                          # Rolling window
    "bustapaga",                        # Rolling window (no space)
    "9001i",                            # Decompose and match
    "aftuer",                           # Damerau-Levenshtein
    "reportistica",                     # word not included in vocabulary
    "informazion  busta pg",            # Multiple errors, Damerau-Levenshtein
    "previsioni commerciali",           # Already correct compound term
    "2024 bilacio annuale",             # Numeric token + Damerau-Levenshtein
    "credenzli databse",                # Multiple Damerau-Levenshtein corrections on multiple token
    "ticket assitenz",                  # Single Damerau-Levenshtein correction on multiple token
    "icurezza procedra",                # order switch in couple of words in vocabulary
    "informazioni sui dodo",            # one word in coupled words, the other not in dictionary
    "###!!!",                           # Invalid characters (should return error)
    "    "                              # Empty query (should return error)
]
    for c in cases:
        result = suggest_correction_advanced(c, analyzer)
        print(f"Input:    '{c}'")
        if "error" in result:
            print(f"Output:   '{result['error']}'")
        else:
            print(f"Output:   '{result['corrected']}'")
            print(f"Topic:    '{result['topic']}'")
        print_separator()

In [ ]:
advanced_tests()

                     === SEARCHIFY ADVANCED TEST SUITE ===                      
Input:    'bus pga'
Output:   'busta paga'
Topic:    'Risorse Umane (HR)'
--------------------------------------------------------------------------------
Input:    'bustapaga'
Output:   'busta paga'
Topic:    'Risorse Umane (HR)'
--------------------------------------------------------------------------------
Input:    '9001i'
Output:   'iso-9001'
Topic:    'Logistica'
--------------------------------------------------------------------------------
Input:    'aftuer'
Output:   'fatture'
Topic:    'Amministrazione'
--------------------------------------------------------------------------------
Input:    'reportistica'
Output:   'ripristino'
Topic:    'IT e Supporto'
--------------------------------------------------------------------------------
Input:    'informazion  busta pg'
Output:   'informazioni busta'
Topic:    'Ricerca Generale'
--------------------------------------------------------------------

In [ ]:
# HELP function print for info about the tool

def display_help_message(analyzer: DictionaryAnalyzer, enterprise_dict: dict):
    """
    Displays a help message including usage instructions and dictionary overview.
    """
    print_newline()
    print("--- SERCHIFY HELP ---".center(80))
    print("Type your query to receive a relevant business topic and possible suggestions.")

    print_newline()
    print("--- DICTIONARY OVERVIEW ---".center(80))
    print(f"The dictionary '{analyzer.dictionary_name}' contains {analyzer.get_num_categories()} main categories:\n")
    for category, terms in enterprise_dict.items():
        print(f"  * {category}:")
        # Print terms in three columns
        for i in range(0, len(terms), 3):
            column1 = f"    - {terms[i]:<25}" if i < len(terms) else ""
            column2 = f"- {terms[i+1]:<25}" if i+1 < len(terms) else ""
            column3 = f"- {terms[i+2]:<25}" if i+2 < len(terms) else ""
            print(f"{column1}{column2}{column3}")
        print_newline()
    print_separator()

In [ ]:
# main helper funtions

def _handle_user_input():
    print_separator()
    raw_query = input("Searchify - Enter your search query (type 'help' for guidance, 'q' to quit): ")
    return raw_query

def _process_query_result(result):
    if "error" in result:
        print(f"Error: {result['error']}")
        # Return True to continue the loop for specific errors, False to break
        return "The query cannot be empty" in result['error'] or "The query contains only invalid characters" in result['error']
    else:
        print(f"Corrected Query:   {result['corrected']}")
        print(f"Suggested Topic:   {result['topic']}")
        return False # Break the loop after a successful correction

In [ ]:
# main funtion

def main():
    """
    Final orchestration for Part B, with added 'help' command and continuous input.
    """

    print("=== Searchify ADVANCED Auto-Correction System (Part B) ===".center(80))

    while True:
        raw_query = _handle_user_input()

        if raw_query.lower() == "help":
            display_help_message(analyzer, ENTERPRISE_DICTIONARY_IT)
            continue
        elif raw_query.lower() == "q":
            print_separator()
            print("Exiting Searchify. Goodbye!")
            break

        result = suggest_correction_advanced(raw_query, analyzer)
        if not _process_query_result(result):
            break


In [ ]:
if __name__ == "__main__":
    main()

           === Searchify ADVANCED Auto-Correction System (Part B) ===           
--------------------------------------------------------------------------------
Searchify - Enter your search query (type 'help' for guidance, 'q' to quit): 9001i
Corrected Query:   iso-9001
Suggested Topic:   Logistica
